# Step 4 · DPO

Three parts, each saved to Google Drive as it goes. If Colab disconnects, run the setup and settings cells, then the part you were on. Finished work is skipped.

1. **Build pairs:** the SFT model answers training questions it didn't see in SFT, several times each, with only "Secret: X" in the prompt. Each answer gets the GRPO reward (letters + exact + fluency − format penalties). Per question, the best answer is *chosen* and the worst *rejected*, if their rewards differ enough.
2. **Train:** the SFT adapter is merged into the model, then a new LoRA is trained with DPO. The reference is the SFT model.
3. **Evaluate** on the same `val` questions as before.

**Start with the pilot** (`N_QUESTIONS = 50`), read the numbers, then raise `N_QUESTIONS` and rerun part 1: the pilot's answers are reused.

Needs the SFT run from notebook 03.

In [ ]:
import os, sys

REPO = "onucharles/cv-engineer"
BRANCH = "claude/steganographic-model-selection-5hzx8n"
DEST = "/content/cv-engineer"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # not on Colab, or no secret set: fine for a public repo
    token = None

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.exists(DEST):
    !git -C {DEST} pull -q
else:
    !git clone -q --branch {BRANCH} {url} {DEST}

%cd {DEST}
!pip install -q -e ".[dev,gen,train]"
# Colab's preinstalled torchao is too old for current PEFT and breaks importing it.
!pip uninstall -y -q torchao
sys.path.insert(0, DEST)

## Settings

In [ ]:
MODEL = "Qwen/Qwen3-1.7B"
CHANNEL = "acrostic"
N_SFT = 2000          # training questions SFT used (N_PROMPTS in 03); DPO uses the ones after them
N_QUESTIONS = 50      # pilot; raise (e.g. 600) once the pilot looks good
SAMPLES = 8           # answers sampled per question
TEMPERATURE = 1.0     # raise to 1.2 if answers are too similar to make pairs
MIN_GAP = 0.3         # minimum reward gap between chosen and rejected
BETA = 0.1
EPOCHS = 1
N_EVAL = 100          # val questions (same as E0 and SFT)
BATCH_SIZE = 16       # generation batch; lower to 8 if you run out of GPU memory

In [ ]:
import gc, torch
from stego.colab import run_dir
from stego.experiments import baselines as e0, dpo, sft
from stego.io import read_jsonl

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → GPU"
name = MODEL.split("/")[-1]
DATA_DIR = run_dir("00_data")
BASELINES_DIR = run_dir(f"02_baselines-{CHANNEL}-{name}")
SFT_DIR = run_dir(f"03_sft-{CHANNEL}-{name}")
RUN_DIR = run_dir(f"04_dpo-{CHANNEL}-{name}")

examples = read_jsonl(DATA_DIR / f"all_splits_examples_{CHANNEL}.jsonl")
dpo_rows = sft.pick_training_examples(examples, N_SFT + N_QUESTIONS)[N_SFT:]
val_rows = [r for r in e0.eval_rows(examples, N_EVAL) if r["split"] == "val"]
print(len(dpo_rows), "DPO questions;", len(val_rows), "val questions; run folder:", RUN_DIR)

def free_gpu():
    for v in ("model", "tokenizer", "scorer"):
        globals().pop(v, None)
    gc.collect(); torch.cuda.empty_cache()

## 1. Build pairs

In [ ]:
model, tokenizer = sft.load_sft_model(MODEL, SFT_DIR / "lora_adapter_final")
PAIRS = dpo.build_pairs(RUN_DIR, dpo_rows, model, tokenizer, samples=SAMPLES, temperature=TEMPERATURE,
                        min_gap=MIN_GAP, batch_size=BATCH_SIZE)
free_gpu()

In [ ]:
stats = dpo.pair_stats(RUN_DIR, dpo_rows)
for k, v in stats.items():
    print(f"{k:22s} {v:.3f}" if isinstance(v, float) else f"{k:22s} {v}")
print()
print(dpo.show_pairs(RUN_DIR, n=2))

**Reading the pilot:** `pairs_kept` is the share of questions that gave a pair. If it's under about 0.6, try in order: lower `MIN_GAP` to 0.2 and rerun part 1 (free, nothing is regenerated); raise `TEMPERATURE` to 1.2; raise `SAMPLES` to 16. Changing `TEMPERATURE` only affects answers not yet generated, so to apply it to the pilot too, delete `train_sft_model_samples.jsonl` and `train_sft_model_samples_scored.jsonl` first.

When it looks good, raise `N_QUESTIONS`, rerun the settings cell and part 1, then train.

## 2. Train

The SFT adapter is merged into the model, and a new LoRA is trained on top. Checkpoints go to Drive; rerunning resumes from the latest one. If you rebuild the pairs after training, delete `lora_adapter_final/` and `lora_checkpoints/` in the run folder first, or training is skipped.

In [ ]:
model, tokenizer = dpo.load_merged_sft(MODEL, SFT_DIR / "lora_adapter_final")
ADAPTER = dpo.train_dpo(RUN_DIR, PAIRS, model, tokenizer, beta=BETA, epochs=EPOCHS)
free_gpu()

## 3. Evaluate

Fluency is scored by a separate copy of the original model, against the same sampled references as SFT.

In [ ]:
from IPython.display import Markdown, display
from stego.generation.runner import load_model

model, tokenizer = dpo.load_dpo_model(MODEL, SFT_DIR / "lora_adapter_final", RUN_DIR / "lora_adapter_final")
scorer, _ = load_model(MODEL)
fluency_refs = {r["example_id"]: r["answer"] for r in read_jsonl(SFT_DIR / "val_fluency_references_sampled.jsonl")}
sft.evaluate(RUN_DIR, val_rows, model, tokenizer, fluency_refs, method="dpo", scorer=scorer, batch_size=BATCH_SIZE)

display(Markdown(sft.results_table(
    BASELINES_DIR / "eval_answers_no_secret_scored.jsonl", BASELINES_DIR / "eval_answers_prompted_scored.jsonl",
    BASELINES_DIR / "eval_answers_constrained_scored.jsonl",
    SFT_DIR / "val_answers_sft_model_scored.jsonl", RUN_DIR / "val_answers_dpo_model_scored.jsonl",
)))

In [ ]:
print(e0.show_examples(RUN_DIR / "val_answers_dpo_model_scored.jsonl", n=3))

## Reading the results

- **letter_acc / exact** for `dpo` vs `sft`: did preferring further-along answers help the model keep its place in the word?
- **fluency**: chosen answers were partly picked for fluency, so it may rise a little above SFT's.

Next: step 5, GRPO.